In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.callbacks import EarlyStopping

print("All libraries imported successfully!")
print("TensorFlow Version:", tf.__version__)

All libraries imported successfully!
TensorFlow Version: 2.21.0


In [3]:
X_train = pd.read_csv("../dataset/processed/X_train_normal_scaled.csv")
X_test = pd.read_csv("../dataset/processed/X_test_scaled.csv")

print("Training Shape:", X_train.shape)
print("Testing Shape:", X_test.shape)

X_train.head()

Training Shape: (37000, 42)
Testing Shape: (175341, 42)


,dur,proto,service,state,spkts,dpkts,sbytes,dbytes,rate,sttl,...,ct_dst_ltm,ct_src_dport_ltm,ct_dst_sport_ltm,ct_dst_src_ltm,is_ftp_login,ct_ftp_cmd,ct_flw_http_mthd,ct_src_ltm,ct_srv_dst,is_sm_ips_ports
0,-0.213727,0.410563,-0.674406,0.932695,-0.124455,-0.151816,-0.043684,-0.087369,0.057181,0.71944,...,-0.563660,-0.468312,-0.450186,-0.477994,-0.090857,-0.090617,-0.203143,-0.640033,-0.644190,-0.10607
1,-0.213728,0.410563,-0.674406,0.932695,-0.124455,-0.151816,-0.036308,-0.087369,0.286565,0.71944,...,-0.563660,-0.468312,-0.450186,-0.477994,-0.090857,-0.090617,-0.203143,-0.640033,-0.644190,-0.10607
2,-0.213729,0.410563,-0.674406,0.932695,-0.124455,-0.151816,-0.040351,-0.087369,0.791209,0.71944,...,-0.563660,-0.468312,-0.450186,-0.390391,-0.090857,-0.090617,-0.203143,-0.640033,-0.554273,-0.10607
3,-0.213729,0.410563,-0.674406,0.932695,-0.124455,-0.151816,-0.041330,-0.087369,0.566923,0.71944,...,-0.444868,-0.349115,-0.450186,-0.390391,-0.090857,-0.090617,-0.203143,-0.522990,-0.554273,-0.10607
4,-0.213728,0.410563,-0.674406,0.932695,-0.124455,-0.151816,-0.034187,-0.087369,0.118350,0.71944,...,-0.444868,-0.349115,-0.450186,-0.390391,-0.090857,-0.090617,-0.203143,-0.522990,-0.554273,-0.10607


In [4]:
y_train = pd.read_csv("../dataset/processed/y_train.csv")
y_test = pd.read_csv("../dataset/processed/y_test.csv")

print(y_train.shape)
print(y_test.shape)

(82332, 1)
(175341, 1)


In [5]:
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(37000, 42)
(175341, 42)
(82332, 1)
(175341, 1)


In [6]:
input_dim = X_train.shape[1]

input_layer = Input(shape=(input_dim,))

# Encoder
encoded = Dense(64, activation="relu")(input_layer)
encoded = Dense(32, activation="relu")(encoded)
encoded = Dense(16, activation="relu")(encoded)
encoded = Dense(8, activation="relu")(encoded)

# Bottleneck
encoded = Dense(4, activation="relu")(encoded)

# Decoder
decoded = Dense(8, activation="relu")(encoded)
decoded = Dense(16, activation="relu")(decoded)
decoded = Dense(32, activation="relu")(decoded)
decoded = Dense(64, activation="relu")(decoded)

decoded = Dense(input_dim, activation="linear")(decoded)

autoencoder = Model(inputs=input_layer, outputs=decoded)

autoencoder.compile(
    optimizer="adam",
    loss="mse"
)

autoencoder.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 42)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         2,752 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 8)              │           136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 4)              │            36 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 8)              │            40 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 16)             │           144 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 32)             │           544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 64)             │         2,112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 42)             │         2,730 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,102 (43.37 KB)

 Trainable params: 11,102 (43.37 KB)

 Non-trainable params: 0 (0.00 B)

In [7]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

history = autoencoder.fit(
    X_train,
    X_train,
    epochs=100,
    batch_size=128,
    validation_split=0.2,
    shuffle=True,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/100
232/232 ━━━━━━━━━━━━━━━━━━━━ 6s 6ms/step - loss: 0.6097 - val_loss: 0.6615
Epoch 2/100
232/232 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.2756 - val_loss: 0.4952
Epoch 3/100
232/232 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.1990 - val_loss: 0.4463
Epoch 4/100
232/232 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.1587 - val_loss: 0.4023
Epoch 5/100
232/232 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.1350 - val_loss: 0.3756
Epoch 6/100
232/232 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.1211 - val_loss: 0.3784
Epoch 7/100
232/232 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.1151 - val_loss: 0.3586
Epoch 8/100
232/232 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.1096 - val_loss: 0.3560
Epoch 9/100
232/232 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.1032 - val_loss: 0.3355
Epoch 10/100
232/232 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.0962 - val_loss: 0.3456
Epoch 11/100
232/232 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.0920 - val_loss: 0.3211
Epoch 12/100
232/232 ━━━━━━━━━━━━━━━━━━━━

In [8]:
autoencoder.save("../results/autoencoder_model.keras")

print("Model saved successfully!")

Model saved successfully!


In [9]:
train_pred = autoencoder.predict(X_train)

1157/1157 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step


In [10]:
train_loss = np.mean(np.square(X_train - train_pred), axis=1)

print("Minimum Train Error:", train_loss.min())
print("Maximum Train Error:", train_loss.max())
print("Average Train Error:", train_loss.mean())

Minimum Train Error: 0.0015838124572448609
Maximum Train Error: 428.0013910764924
Average Train Error: 0.07838932083311688


In [48]:
from sklearn.metrics import f1_score
import numpy as np

thresholds = np.percentile(train_loss, np.arange(90, 100, 0.5))

best_threshold = None
best_f1 = 0

for t in thresholds:
    preds = (test_loss > t).astype(int)
    score = f1_score(y_test, preds)

    if score > best_f1:
        best_f1 = score
        best_threshold = t

print("Best Threshold:", best_threshold)
print("Best F1 Score:", best_f1)

Best Threshold: 0.5330228854400606
Best F1 Score: 0.8178298292261725


In [49]:
threshold = best_threshold

print("Threshold:", threshold)

Threshold: 0.5330228854400606


In [27]:
test_pred = autoencoder.predict(X_test)

5480/5480 ━━━━━━━━━━━━━━━━━━━━ 9s 2ms/step


In [39]:
test_loss = np.mean(np.square(X_test - test_pred), axis=1)

print("Minimum Test Error:", test_loss.min())
print("Maximum Test Error:", test_loss.max())
print("Average Test Error:", test_loss.mean())

Minimum Test Error: 0.016484075401288822
Maximum Test Error: 2824.255189799405
Average Test Error: 1.4957288967609688


In [40]:
test_predictions = (test_loss > threshold).astype(int)

print(test_predictions[:20])

0     1
1     1
2     1
3     1
4     1
5     1
6     1
7     1
8     1
9     1
10    1
11    1
12    1
13    1
14    1
15    1
16    1
17    1
18    1
19    1
dtype: int64


In [50]:
from sklearn.metrics import confusion_matrix, classification_report

cm = confusion_matrix(y_test, test_predictions)

print(cm)

print("\nClassification Report:\n")
print(classification_report(y_test, test_predictions))

[[  2834  53166]
 [     0 119341]]

Classification Report:

              precision    recall  f1-score   support

           0       1.00      0.05      0.10     56000
           1       0.69      1.00      0.82    119341

    accuracy                           0.70    175341
   macro avg       0.85      0.53      0.46    175341
weighted avg       0.79      0.70      0.59    175341



In [32]:
print("Min:", np.min(train_loss))
print("Max:", np.max(train_loss))

print("90th:", np.percentile(train_loss, 90))
print("95th:", np.percentile(train_loss, 95))
print("97th:", np.percentile(train_loss, 97))
print("99th:", np.percentile(train_loss, 99))
print("99.5th:", np.percentile(train_loss, 99.5))

Min: 0.0015838124572448609
Max: 428.0013910764924
90th: 0.09710789800679051
95th: 0.13481503833898387
97th: 0.16984954299881574
99th: 0.3517113052308823
99.5th: 0.5330228854400606


In [33]:
print(type(train_loss))
print(train_loss.shape)

<class 'pandas.Series'>
(37000,)


In [18]:
print(y_train["label"].value_counts())

label
1    45332
0    37000
Name: count, dtype: int64


In [19]:
X_train_normal = pd.read_csv("../dataset/processed/X_train_normal.csv")
print(X_train_normal.shape)
X_train_normal.head()

(37000, 42)


,dur,proto,service,state,spkts,dpkts,sbytes,dbytes,rate,sttl,...,ct_dst_ltm,ct_src_dport_ltm,ct_dst_sport_ltm,ct_dst_src_ltm,is_ftp_login,ct_ftp_cmd,ct_flw_http_mthd,ct_src_ltm,ct_srv_dst,is_sm_ips_ports
0,0.000011,117.0,0.0,4.0,2,0,496,0,90909.0902,254,...,1,1,1,2,0,0,0,1,2,0
1,0.000008,117.0,0.0,4.0,2,0,1762,0,125000.0003,254,...,1,1,1,2,0,0,0,1,2,0
2,0.000005,117.0,0.0,4.0,2,0,1068,0,200000.0051,254,...,1,1,1,3,0,0,0,1,3,0
3,0.000006,117.0,0.0,4.0,2,0,900,0,166666.6608,254,...,2,2,1,3,0,0,0,2,3,0
4,0.000010,117.0,0.0,4.0,2,0,2126,0,100000.0025,254,...,2,2,1,3,0,0,0,2,3,0


In [51]:
print("Threshold:", threshold)

Threshold: 0.5330228854400606


In [52]:
import joblib

joblib.dump(best_threshold, "../results/best_threshold.pkl")

print("Best threshold saved.")

Best threshold saved.
